In [ ]:
# === mount Drive ===
from google.colab import drive
drive.mount('/content/drive')

# === settings ===
import os
import yaml, copy
import pandas as pd, numpy as np

BASE_DIR = '/content/drive/MyDrive/SCMLLM' # path to 'My Drive'

CONFIG_PATH = os.path.join(BASE_DIR, "config.yaml")
DATA_PATH   = os.path.join(BASE_DIR, "finalized_dynamic_supply_chain_logistics.csv")
OUTPUT_CSV  = os.path.join(BASE_DIR, "sensitivity_layer1.csv")

BASE_CONFIG = yaml.safe_load(open(CONFIG_PATH))
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} records")

RECORDS  = df.to_dict(orient='list')
NAN_MASK = {c: pd.isna(df[c]).values for c in df.columns}
N = len(df)
BOUNDED_FEATURES = frozenset([
    'supplier_reliability_score', 'weather_condition_severity',
    'driver_behavior_score', 'fatigue_monitoring_score',
])

# === deterministic evaluation ===
def evaluate_dataset(config):
    agent_names = [k for k in config if k.startswith('agent_')]
    w   = config['supervisor']['agent_weights']
    thr = config['supervisor']['risk_thresholds']
    flagged = np.zeros(N, bool); gs = np.zeros(N, float)
    for an in agent_names:
        ac = config[an]; af = np.zeros(N, bool); asc = np.zeros(N, float)
        for rule in ac['rules']:
            f = rule['feature']
            if f not in RECORDS: continue
            col = np.asarray(RECORDS[f], dtype=float); valid = ~NAN_MASK[f]
            t = rule['threshold']; c = rule['condition']
            if   c == 'greater_than':  tr = valid & (col > t)
            elif c == 'less_than':     tr = valid & (col < t)
            elif c == 'outside_range': tr = valid & ((col < t[0]) | (col > t[1]))
            elif c == 'equals':        tr = valid & (col == t)
            else:                      tr = np.zeros(N, bool)
            af |= tr
        flagged |= af
        for sr in ac['scoring']:
            f = sr['feature']
            if f not in RECORDS: continue
            raw = RECORDS[f]; valid = ~NAN_MASK[f]; ct = sr['type']; s = np.zeros(N, float)
            if ct == 'direct':
                col = np.asarray(raw, dtype=float); s[valid] = col[valid]
            elif ct == 'categorical':
                m = sr['mapping']; ca = np.asarray(raw)
                for idx in np.where(valid)[0]: s[idx] = float(m.get(ca[idx], 0.0))
            elif ct in ('normalize', 'inverse_normalize'):
                col = np.asarray(raw, dtype=float); mn, mx = sr['range']
                if mx > mn:
                    cl = np.clip(col, mn, mx); nm = (cl - mn) / (mx - mn)
                    s[valid] = (1.0 - nm[valid]) if ct == 'inverse_normalize' else nm[valid]
            elif ct == 'deviation_normalize':
                col = np.asarray(raw, dtype=float); smn, smx = sr['safe_range']; bmn, bmx = sr['bounds']
                below = valid & (col < smn); above = valid & (col > smx)
                if (smn - bmn) > 0: s[below] = np.minimum(1.0, (smn - col[below]) / (smn - bmn))
                if (bmx - smx) > 0: s[above] = np.minimum(1.0, (col[above] - smx) / (bmx - smx))
            asc = np.maximum(asc, s)
        gs += asc * w.get(an, 0.0)
    classes = np.full(N, 'LOW', dtype=object)
    classes[gs >= thr['MODERATE']] = 'MODERATE'
    classes[gs >= thr['HIGH']]     = 'HIGH'
    return flagged, gs, classes.tolist()

def run_scenario(name, config, base=None):
    fl, sc, cl = evaluate_dataset(config)
    vc = pd.Series(cl).value_counts(normalize=True) * 100
    stab = 100.0
    if base is not None:
        stab = (np.sum(np.asarray(cl) == np.asarray(base)) / N) * 100
    return {'Perturbation': name, 'Flag rate (%)': round(fl.mean() * 100, 2),
            'Mean S_global': round(sc.mean(), 4),
            'Low/Mod/High (%)': f"{vc.get('LOW',0):.1f} / {vc.get('MODERATE',0):.1f} / {vc.get('HIGH',0):.1f}",
            'Class stability (%)': round(stab, 2), '_cl': cl}

# === perturbation generators ===
def apply_magnitude_shift(config, shift):
    nc = copy.deepcopy(config)
    for a, d in nc.items():
        if not a.startswith('agent_'): continue
        for rule in d['rules']:
            if rule['condition'] == 'equals' or rule['feature'] in BOUNDED_FEATURES: continue
            if isinstance(rule['threshold'], list): rule['threshold'] = [v*(1+shift) for v in rule['threshold']]
            else: rule['threshold'] *= (1+shift)
    return nc

def apply_scoring_shift(config, shift):
    nc = copy.deepcopy(config)
    for a, d in nc.items():
        if not a.startswith('agent_'): continue
        for sr in d.get('scoring', []):
            if sr['feature'] in BOUNDED_FEATURES: continue
            for k in ('range', 'safe_range', 'bounds'):
                if k in sr: sr[k] = [v*(1+shift) for v in sr[k]]
    return nc

def apply_cutoff_shift(config, shift):
    nc = copy.deepcopy(config)
    nc['supervisor']['risk_thresholds']['HIGH']     += shift
    nc['supervisor']['risk_thresholds']['MODERATE'] += shift
    return nc

def apply_equal_weights(config):
    nc = copy.deepcopy(config)
    for a in nc['supervisor']['agent_weights']: nc['supervisor']['agent_weights'][a] = 1.0/6.0
    return nc

# === run all scenarios  ===
SCENARIOS = [
    ('Baseline',            BASE_CONFIG),
    ('Thresholds +10%',     apply_magnitude_shift(BASE_CONFIG,  0.10)),
    ('Thresholds +20%',     apply_magnitude_shift(BASE_CONFIG,  0.20)),
    ('Thresholds -10%',     apply_magnitude_shift(BASE_CONFIG, -0.10)),
    ('Thresholds -20%',     apply_magnitude_shift(BASE_CONFIG, -0.20)),
    ('Equal Weights',       apply_equal_weights(BASE_CONFIG)),
    ('Cut-offs +0.05',      apply_cutoff_shift(BASE_CONFIG,  0.05)),
    ('Cut-offs +0.10',      apply_cutoff_shift(BASE_CONFIG,  0.10)),
    ('Cut-offs -0.05',      apply_cutoff_shift(BASE_CONFIG, -0.05)),
    ('Cut-offs -0.10',      apply_cutoff_shift(BASE_CONFIG, -0.10)),
    # scoring-range scenarios: the parameters that actually drive the score
    ('Scoring ranges +10%', apply_scoring_shift(BASE_CONFIG,  0.10)),
    ('Scoring ranges -10%', apply_scoring_shift(BASE_CONFIG, -0.10)),
    ('Scoring ranges +20%', apply_scoring_shift(BASE_CONFIG,  0.20)),
    ('Scoring ranges -20%', apply_scoring_shift(BASE_CONFIG, -0.20)),
]

base = run_scenario('Baseline', BASE_CONFIG)
bc = base['_cl']
results = [base] + [run_scenario(n, c, bc) for n, c in SCENARIOS[1:]]

print(f"\n{'Perturbation':<22}{'Flag%':>8}{'MeanS':>9}   {'Low/Mod/High':<20}{'Stab%':>8}")
print('-' * 72)
for r in results:
    print(f"{r['Perturbation']:<22}{r['Flag rate (%)']:>8}{r['Mean S_global']:>9}   "
          f"{r['Low/Mod/High (%)']:<20}{r['Class stability (%)']:>8}")

pd.DataFrame([{k: v for k, v in r.items() if k != '_cl'} for r in results]).to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved to {OUTPUT_CSV}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 32065 records

Perturbation             Flag%    MeanS   Low/Mod/High           Stab%
------------------------------------------------------------------------
Baseline                 99.96   0.7423   0.1 / 35.0 / 64.9      100.0
Thresholds +10%          99.97   0.7423   0.1 / 35.0 / 64.9      100.0
Thresholds +20%          99.95   0.7423   0.1 / 35.0 / 64.9      100.0
Thresholds -10%          99.99   0.7423   0.1 / 35.0 / 64.9      100.0
Thresholds -20%          100.0   0.7423   0.1 / 35.0 / 64.9      100.0
Equal Weights            99.96   0.7529   0.0 / 31.9 / 68.1      89.47
Cut-offs +0.05           99.96   0.7423   0.3 / 49.0 / 50.7      85.59
Cut-offs +0.10           99.96   0.7423   0.9 / 63.3 / 35.8       70.1
Cut-offs -0.05           99.96   0.7423   0.0 / 23.8 / 76.2      88.58
Cut-offs -0.10           99.96   0.7423   0.0 / 15.1 / 84.9      7